In [ ]:
CATALOG = "demo_medallion"
BRONZE_SCHEMA = "bronze"

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")

print("Catalog and Bronze schema are ready.")

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

customer_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("status", StringType(), True)
])

customer_data = [
    (101, "Rahul Sharma", "rahul@gmail.com", "Mumbai", "ACTIVE"),
    (102, "Priya Patil", "priya@gmail.com", "Pune", "ACTIVE"),
    (103, "Amit Shah", "amit@gmail.com", "Ahmedabad", "ACTIVE"),
    (104, "Neha Joshi", "neha@gmail.com", "Nashik", "INACTIVE"),
    (105, "Vikas Mehta", "vikas@gmail.com", "Delhi", "ACTIVE"),

    # Duplicate
    (103, "Amit Shah", "amit@gmail.com", "Ahmedabad", "ACTIVE"),

    # Missing email
    (106, "Sneha Deshmukh", None, "Pune", "ACTIVE"),

    # Invalid status
    (107, "Rohit Kulkarni", "rohit@gmail.com", "Mumbai", "UNKNOWN")
]

df_customers = spark.createDataFrame(
    customer_data,
    customer_schema
)

display(df_customers)

In [ ]:
CUSTOMER_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.customers"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CUSTOMER_TABLE} (
    customer_id INT,
    customer_name STRING,
    email STRING,
    city STRING,
    status STRING
)
USING DELTA
""")

In [ ]:
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(CUSTOMER_TABLE)

print(f"Bronze table created/updated: {CUSTOMER_TABLE}")